# PT-LLMvsPy — Notebook 02 : audit du nettoyage et description lexicale

**Question provisoire :** quelles opérations de prétraitement ont été effectivement exécutées sur *Le Féminisme*, lesquelles restent à définir, et quels résultats lexicométriques peut-on calculer à partir du texte nettoyé ?

Ce notebook reprend les **douze opérations OP01–OP12** du tableau de spécification (voir `docs/matrice_pretraitements_gemini.md`) et les indicateurs du rapport *Rapport d’analyse lexicométrique, quantitative et relationnelle de l’ouvrage Le Féminisme* fourni pour le projet.

**Enchaînement :** ouvrir le texte nettoyé par le notebook 01 → vérifier les opérations exécutées → tester leur idempotence → inventorier les traitements manquants → calculer les métriques globales, champs lexicaux, marqueurs énonciatifs, expressions et mentions de personnes → enregistrer les résultats.

Les **cooccurrences phrastiques et en fenêtres ±5/±10 tokens** relèveront du notebook 03. Elles doivent partir du **même corpus préparé**.

## 0 — Retrouver la source nettoyée

Le notebook 01 a produit `faguet_nettoye.txt` et `metadonnees_analyse.json` dans `/content/analyse_faguet_v2_0_0`. Le second fichier contient les opérations réellement exécutées et l'empreinte du premier.

**Ne retéléchargez pas le texte brut pour remplacer le fichier nettoyé.** Si l'environnement Colab a été réinitialisé, recopiez le dossier depuis votre Drive ou adaptez `DOSSIER_N1` à son emplacement monté.

In [ ]:
import hashlib, importlib, json, secrets, subprocess, sys, re, zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen
from urllib.parse import quote

DEPOT_LIB = "dreymond732/tal-notebook-evaluator"
REVISION_GIT = "main"
VERSION_NOTEBOOK_02 = "1.0.0"
DOSSIER_N1 = Path("/content/analyse_faguet_v2_0_0")
DOSSIER_N2 = Path("/content/analyse_faguet_n2_v" + VERSION_NOTEBOOK_02.replace(".", "_"))
if not Path("/content").exists():
    DOSSIER_N1 = Path.cwd() / DOSSIER_N1.name
    DOSSIER_N2 = Path.cwd() / DOSSIER_N2.name
DOSSIER_N2.mkdir(parents=True, exist_ok=True)

url_commit = ("https://api.github.com/repos/" + DEPOT_LIB
              + "/commits/" + quote(REVISION_GIT, safe=""))
with urlopen(url_commit, timeout=30) as r:
    COMMIT_LIB = json.loads(r.read().decode("utf-8"))["sha"]
assert len(COMMIT_LIB) == 40
racine = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd()/"PT-LLMvsPy")
               if (p/"src/tal_toolkit/analyse_pt02.py").exists()), None)
if racine:
    sys.path.insert(0, str(racine/"src"))
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "--force-reinstall",
        f"git+https://github.com/{DEPOT_LIB}.git@{COMMIT_LIB}#subdirectory=PT-LLMvsPy"
    ])
importlib.invalidate_caches()
try:
    import spacy
    from packaging.version import Version
    if Version(spacy.__version__) < Version("3.8"):
        raise ImportError("spaCy ancien")
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","spacy>=3.8,<4"])
    import spacy
import pandas as pd
from tal_toolkit import (charger_corpus, recenser_elements,
    OperationPretraitement, PlanPretraitement,
    ProtocoleGlobal, calculer_metriques_globales, comparer_annonces)
from tal_toolkit.analyse_pt02 import (
    auditer_reapplication, analyser_lexique_pt02, CHAMPS_LEXICAUX)
print("Dossier du notebook 01 :", DOSSIER_N1)
print("Révision Git des fonctions :", COMMIT_LIB)

In [ ]:
FICHIER_SOURCE = DOSSIER_N1 / "faguet_nettoye.txt"
FICHIER_META_N1 = DOSSIER_N1 / "metadonnees_analyse.json"
if not FICHIER_SOURCE.is_file() or not FICHIER_META_N1.is_file():
    raise FileNotFoundError(
        "Il faut le texte nettoyé ET metadonnees_analyse.json du notebook 01. "
        "Recopiez le dossier depuis Drive ou modifiez DOSSIER_N1."
    )
META_N1 = json.loads(FICHIER_META_N1.read_text(encoding="utf-8"))
corpus = charger_corpus(FICHIER_SOURCE, sha256_attendu=META_N1["sha256_nettoye"])
print("Corpus nettoyé :", len(corpus.texte), "caractères")
print("SHA-256 :", corpus.sha256)
print("Notebook 01 :", META_N1["version_notebook"])
print("Fonctions notebook 01 :", META_N1["revision_git"])

## 1 — Quelles opérations sont déjà réalisées ?

La matrice de spécification distingue les **transformations du texte** des **annotations et règles de mesure**.

| Op. | Proposition | Situation à vérifier |
|---|---|---|
| OP01 | Péritexte, en-têtes, séparateurs | Séparateurs masqués ; titres conservés |
| OP02 | Appels et blocs de notes | Masquage effectué |
| OP03 | Recoller les césures de fin de ligne | Non activé automatiquement |
| OP04 | Recomposer les retours typographiques | Effectué dans le notebook 01 |
| OP05 | Harmoniser les apostrophes | Fonction disponible, non activée |
| OP06 | Traits d'union et composés | Tokenisation à examiner, sans réécriture |
| OP07 | Minuscules | Normalisation **des formes comptées**, pas du texte |
| OP08 | Tokenisation | Annotation spaCy, pas une transformation éditoriale |
| OP09 | Filtrage des tokens | Alphabétiques retenus ; mots vides conservés |
| OP10 | Lemmatisation | Nécessite un modèle annotant effectivement |
| OP11 | Segmentation en phrases | spaCy `sentencizer` dans le premier notebook |
| OP12 | Protection des abréviations | Non implémentée spécifiquement |

La cellule ci-dessous lit le **journal du premier traitement**. Seule cette trace permet d'indiquer combien d'occurrences ont été effectivement modifiées.

In [ ]:
journal_operations = pd.DataFrame([
    {"operation": ligne["operation"], "modifications": ligne["modifications"]}
    for ligne in META_N1["operations"]
])
display(journal_operations)
inventaire = recenser_elements(corpus, max_exemples=4)
display(pd.DataFrame([
    {"categorie": nom, "elements_restants": infos["nombre"]}
    for nom, infos in inventaire["categories"].items()
]))
print("Un élément encore présent n'est pas nécessairement une erreur.")

### 1.1 — Réapplication et idempotence

Pour un plan de traitement `F`, on teste la propriété **F(F(x)) = F(x)** : après un premier passage, un deuxième passage devrait idéalement ne plus modifier le texte. Il ne faut pas confondre cela avec la fonction identité **Id(x) = x**.

La vérification suivante rejoue **sur une copie** les opérations décrites dans les métadonnées du notebook 01, sans toucher au texte utilisé dans les mesures. Si un écart apparaît, il est affiché pour examen.

In [ ]:
plan_source = META_N1["plan_pretraitement"]
plan_rejoue = PlanPretraitement(tuple(
    OperationPretraitement(op["nom"], op["justification"], tuple(op.get("cibles", ())))
    for op in plan_source["operations"]
), identifiant=plan_source["identifiant"])
bilan_idempotence = auditer_reapplication(corpus.texte, plan_rejoue)
print("Idempotent sur ce texte :", bilan_idempotence["idempotent_sur_ce_texte"])
display(pd.DataFrame(bilan_idempotence["operations"]))
if not bilan_idempotence["idempotent_sur_ce_texte"]:
    print("Premier écart :", bilan_idempotence["premier_ecart_position"])
    print("Avant :", repr(bilan_idempotence["avant"]))
    print("Après :", repr(bilan_idempotence["apres"]))
print("Le fichier nettoyé demeure inchangé :", corpus.sha256)

### 1.2 — Opérations restant à définir

- **OP03 :** rechercher des césures candidates, mais n'en recoller aucune sans liste de cas validée.
- **OP05–OP06 :** inventorier apostrophes et tirets ; distinguer leur graphie des frontières de tokens.
- **OP10 :** la lemmatisation n'est pas disponible avec un simple `spacy.blank("fr")`. Utiliser un modèle annotant avant de produire des comptes de *lemmes*.
- **OP12 :** inspecter les abréviations (`M.`, `Mme`, `MM.`, `Dr`, etc.) et leurs conséquences possibles sur les phrases.

Ces décisions n'empêchent pas de calculer les mesures de **formes minuscules** sur la version nettoyée actuellement disponible. Aucune transformation supplémentaire n'est appliquée implicitement.

In [ ]:
residus = {
    "apostrophes_courbes": corpus.texte.count("’"),
    "tirets_cadratins": corpus.texte.count("—"),
    "demi_cadratins": corpus.texte.count("–"),
    "cesures_candidates": inventaire["categories"]["cesures_candidates"]["nombre"],
}
display(pd.DataFrame([{"element": k, "nombre": v} for k, v in residus.items()]))
tokeniseur = spacy.blank("fr")
for exemple in ("lui-même", "l'homme", "aujourd'hui", "M.", "Mme"):
    print(exemple, "→", tokeniseur.tokenizer.explain(exemple))

## 2 — Calculer les métriques globales (partie 1 du rapport)

Le rapport propose sept indicateurs : tokens bruts, tokens lexicaux, types, hapax, phrases, TTR et longueur moyenne des phrases. Il demande aussi un **STTR par blocs de 1 000 tokens**, absent du script final : nous le calculons.

Par défaut, on compte les tokens `is_alpha`, les formes minuscules et les phrases non vides identifiées par le `sentencizer`. Les mots vides sont conservés. Un modèle statistique français peut être choisi explicitement, sans repli silencieux qui fabriquerait des lemmes.

In [ ]:
MODELE_SPACY = "blank-fr"  # Facultatif : modèle installé, ex. "fr_core_news_sm"
LEMMES_DEMANDES = MODELE_SPACY != "blank-fr"
TAILLE_STTR = 1000
if MODELE_SPACY == "blank-fr":
    nlp = spacy.blank("fr")
else:
    try:
        nlp = spacy.load(MODELE_SPACY)
    except OSError as exc:
        raise RuntimeError("Modèle absent : installez-le ou choisissez 'blank-fr'.") from exc
if not any(nlp.has_pipe(k) for k in ("parser", "senter", "sentencizer")):
    nlp.add_pipe("sentencizer")
nlp.max_length = max(nlp.max_length, len(corpus.texte) + 1000)
resultats_pt02 = analyser_lexique_pt02(
    corpus.texte, nlp, taille_sttr=TAILLE_STTR,
    calculer_lemmes=LEMMES_DEMANDES
)
assert resultats_pt02["source_nettoyee_sha256"] == corpus.sha256
assert all(resultats_pt02["controles"].values())
print("Lemmatisation :", resultats_pt02["lemmatisation"]["statut"])
print("Pipeline :", nlp.pipe_names)

In [ ]:
g = resultats_pt02["globales"]
INDICATEURS = {
    "METR_01_TOK_BRUT": ("tokens_bruts_spacy", "tous tokens spaCy"),
    "METR_02_TOK_LEX": ("tokens_lexicaux", "tokens alphabétiques"),
    "METR_03_TYPES_V": ("types_minuscules", "types en minuscules"),
    "METR_04_HAPAX": ("hapax", "formes présentes une fois"),
    "METR_05_SENT_COUNT": ("phrases_non_vides", "phrases avec mots"),
    "METR_06_TTR": ("ttr", "types / tokens"),
    "METR_07_L_MOY_SENT": ("longueur_moyenne_phrase", "tokens / phrase"),
    "METR_08_STTR": ("sttr", "TTR moyen par blocs complets"),
}
tableau_global = pd.DataFrame([
    {"identifiant": k, "definition": titre,
     "valeur": g[code], "statut": "CALCULÉ" if g[code] is not None else "NON_CALCULÉ"}
    for k, (code, titre) in INDICATEURS.items()
])
display(tableau_global)
print("STTR : taille des blocs =", TAILLE_STTR,
      ", blocs complets =", g["sttr_nombre_blocs_complets"],
      ", reliquat non utilisé =", g["sttr_reliquat_exclu"])

### 2.1 — Contrôle croisé des mesures et annonces Gemini

Le moteur du semestre permet de **recalculer indépendamment** les comptes principaux sur le même texte. La comparaison avec les nombres annoncés par Gemini est une opération distincte : un écart numérique ne démontre pas, à lui seul, un dysfonctionnement.

In [ ]:
if MODELE_SPACY == "blank-fr":
    convention = ProtocoleGlobal(
        tokens="alphabetiques", formes="minuscules", retours="conserver",
        ignorer_phrases_sans_mots=True
    )
    autre_mesure = calculer_metriques_globales(
        corpus, nlp, protocole=convention, max_exemples=4
    )
    assert autre_mesure["metriques"]["tokens"] == g["tokens_lexicaux"]
    assert autre_mesure["metriques"]["types"] == g["types_minuscules"]
    assert autre_mesure["metriques"]["hapax"] == g["hapax"]
    assert autre_mesure["metriques"]["phrases"] == g["phrases_non_vides"]
    print("Contrôle interne des quatre effectifs : OK")
else:
    autre_mesure = None
    print("Le modèle expérimental doit être documenté séparément.")

adresse_gemini = ("https://raw.githubusercontent.com/" + DEPOT_LIB + "/"
                 + COMMIT_LIB + "/PT-LLMvsPy/data/annonces_globales_gemini.json")
with urlopen(adresse_gemini, timeout=30) as r:
    ANN_GEMINI = json.loads(r.read().decode("utf-8"))
if autre_mesure:
    tableau_gemini = pd.DataFrame(
        comparer_annonces(autre_mesure, ANN_GEMINI["annonces"])
    )
    display(tableau_gemini[[
        "cle", "valeur_reference", "observe_brut", "statut_numerique"
    ]])
else:
    tableau_gemini = None

## 3 — Champs lexicaux (partie 2 du rapport)

Les listes du rapport définissent cinq champs **non exclusifs** : genre, droit, mœurs/famille, travail/éducation, féminisme/idéologies. Les valeurs ci-dessous comptent d'abord les **formes minuscules exactes**. `femme` et `femmes` ne sont donc pas automatiquement regroupés.

Les comptes par **lemmes** sont présentés séparément, uniquement si un lemmatiseur effectif a été choisi. Les ambiguïtés de `droit` ou `femme` exigent toujours une lecture contextuelle.

In [ ]:
tableau_champs = pd.DataFrame([
    {"champ": nom, "occurrences_formes": d["occurrences"],
     "part_tokens_pct": d["part_tokens_lexicaux_pct"],
     "termes_definis": len(CHAMPS_LEXICAUX[nom])}
    for nom, d in resultats_pt02["champs_lexicaux"].items()
])
display(tableau_champs)
if resultats_pt02["lemmatisation"]["statut"] == "CALCULE":
    display(pd.DataFrame([
        {"champ": n, "occurrences_lemmes": d["occurrences"]}
        for n, d in resultats_pt02["lemmatisation"]["champs"].items()
    ]))
else:
    print("Champs par lemmes : NON CALCULÉS.")

## 4 — Marqueurs énonciatifs, locutions et personnes (parties 3 et 5)

Le script du rapport propose les formes de première personne, l'adresse à l'interlocuteur, les expressions `il faut`, `je crois`, `égalité des sexes` ou `division du travail`, ainsi que les mentions de personnes.

Nous comptons des **graphies et séquences explicites**, sans inférer une valeur rhétorique ni confondre une mention orthographique et une entité désambiguïsée. Les exemples conservés donnent des positions **dans le texte nettoyé**.

In [ ]:
e = resultats_pt02["enonciation"]
display(pd.DataFrame([
    {"forme": k, "1re_personne": v} for k, v in
    e["premiere_personne_formes"].items()
]))
display(pd.DataFrame([
    {"forme": k, "adresse": v} for k, v in e["adresse_formes"].items()
]))
tableau_locutions = pd.DataFrame([
    {"expression": nom, "occurrences": v["occurrences"]}
    for nom, v in resultats_pt02["locutions"].items()
])
tableau_personnes = pd.DataFrame([
    {"personne": nom, "mentions_orthographiques": v["mentions"]}
    for nom, v in resultats_pt02["onomastique"].items()
])
display(tableau_locutions)
display(tableau_personnes)
for nom in ("il_faut", "je_crois"):
    for preuve in resultats_pt02["locutions"][nom]["preuves"][:2]:
        print(nom, preuve["debut"], repr(preuve["extrait"][:160]))

### Examiner les abréviations (OP12)

Le `sentencizer` peut traiter incorrectement certaines abréviations (par exemple `M.`, `Mme` ou `MM.`). Les sites ci-dessous constituent des **candidats à inspecter**, non un décompte d'erreurs prouvées.

In [ ]:
abreviations = []
for motif in (r"\bM\.", r"\bMM\.", r"\bMme\b", r"\bMlle\b", r"\bDr\b"):
    elements = list(re.finditer(motif, corpus.texte))
    abreviations.append({
        "motif": motif,
        "nombre": len(elements),
        "premier_contexte": (
            corpus.texte[max(0,elements[0].start()-45):elements[0].end()+65]
            if elements else ""
        ),
    })
display(pd.DataFrame(abreviations))

## 5 — Sauvegarder les résultats et la provenance

Les tableaux, positions de preuve, paramètres et références au dossier du notebook 01 sont enregistrés dans `/content/analyse_faguet_n2_v1_0_0`. Aucune validation manuelle ni valeur chiffrée préremplie n'est nécessaire.

In [ ]:
def enregistrer(df, nom):
    fichier = DOSSIER_N2 / nom
    df.to_csv(fichier, index=False, encoding="utf-8")
    return fichier

fichiers = [
    enregistrer(tableau_global, "metriques_globales.csv"),
    enregistrer(tableau_champs, "champs_lexicaux.csv"),
    enregistrer(tableau_locutions, "locutions.csv"),
    enregistrer(tableau_personnes, "mentions_personnes.csv"),
]
if tableau_gemini is not None:
    fichiers.append(enregistrer(tableau_gemini, "comparaison_gemini.csv"))
dossier_preuves = {
    "schema": "pt02-preuves-v1",
    "version_notebook_02": VERSION_NOTEBOOK_02,
    "revision_bibliotheque_02": COMMIT_LIB,
    "source_notebook_01": str(FICHIER_SOURCE),
    "metadonnees_notebook_01": META_N1,
    "sha256_texte_nettoye": corpus.sha256,
    "reapplication": bilan_idempotence,
    "inventaire_residuel": inventaire,
    "resultats": resultats_pt02,
}
FICHIER_PREUVES = DOSSIER_N2 / "resultats_et_preuves.json"
FICHIER_PREUVES.write_text(
    json.dumps(dossier_preuves, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)
fichiers.append(FICHIER_PREUVES)
# Garder avec les résultats l'exemplaire exact de la source nettoyée
# et les paramètres produits par le notebook 01.
fichiers.extend([FICHIER_SOURCE, FICHIER_META_N1])
date = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
numero = f"{secrets.randbelow(1_000_000_000):09d}"
ARCHIVE_N2 = DOSSIER_N2 / f"FAGUET_N2_{date}_{numero}.zip"
with zipfile.ZipFile(ARCHIVE_N2, "w", zipfile.ZIP_DEFLATED) as z:
    for fichier in fichiers:
        z.write(fichier, arcname=fichier.name)
with zipfile.ZipFile(ARCHIVE_N2) as z:
    assert z.testzip() is None
print("Résultats :", DOSSIER_N2)
print("Archive :", ARCHIVE_N2)

### Conserver une copie dans Drive

Comme pour le premier notebook, montez votre Drive dans Colab, repérez le dossier partagé et renseignez son chemin ci-dessous. La copie est facultative et ne remplace pas un fichier de même nom.

In [ ]:
DOSSIER_DEPOT_DRIVE = ""
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Montage Drive disponible uniquement dans Colab.")
if DOSSIER_DEPOT_DRIVE.strip():
    from shutil import copy2
    cible = Path(DOSSIER_DEPOT_DRIVE)
    if not cible.is_dir():
        raise FileNotFoundError("Dossier Drive introuvable.")
    depot = cible / ARCHIVE_N2.name
    if depot.exists():
        raise FileExistsError("Archive déjà présente : aucun écrasement.")
    copy2(ARCHIVE_N2, depot)
    assert hashlib.sha256(ARCHIVE_N2.read_bytes()).digest() == hashlib.sha256(
        depot.read_bytes()
    ).digest()
    print("Dépôt vérifié :", depot)
else:
    print("Archive locale disponible :", ARCHIVE_N2)

## Suite de l'analyse

À examiner collectivement : les éventuelles modifications après réapplication, les choix sur les césures, apostrophes et tirets, la validité de la segmentation en phrases et la qualité d'une éventuelle lemmatisation.

Les **cooccurrences par phrase et fenêtres ±5/±10** sont réservées au notebook 03 ; l'audit des réponses LLM et de leurs citations au notebook 04. Ces analyses devront partager **le même texte nettoyé**, sans reprendre silencieusement le texte original.

*Références* : matrice OP01–OP12 du dépôt ; rapport Faguet fourni ; Manning, Raghavan et Schütze (2008), chap. 2 ; Bird, Klein et Loper (2009), chap. 3 ; Evert (2009), « Corpora and collocations » (manuscrit : https://stephanie-evert.de/PUB/Evert2007HSK_extended_manuscript.pdf).